# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Wodecki Twist Festiwal 2026
2. Że co, ja nie dam rady? – komediowe show improwizowane
3. Mayday odNowa
4. O mało co...
5. Trzy siostry (sztuka na wynos)
6. Miesiąc Fotografii 2026
7. XXXII Międzynarodowy Festiwal „Starzy i Młodzi, czyli Jazz w Krakowie”
8. Andrzej Dudziński. Przypadek artysty niemożliwego
9. Teatro Español w Madrycie. Najstarszy teatr w Europie 1583-2026
10. Piąty element – MIŁOŚĆ. Wystawa Krakowskiego Klubu Fotograficznego

Czas wykonania: 3.00s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Wodecki Twist Festiwal 2026
2. Że co, ja nie dam rady? – komediowe show improwizowane
3. Mayday odNowa
4. O mało co...
5. Trzy siostry (sztuka na wynos)
6. Miesiąc Fotografii 2026
7. XXXII Międzynarodowy Festiwal „Starzy i Młodzi, czyli Jazz w Krakowie”
8. Andrzej Dudziński. Przypadek artysty niemożliwego
9. Teatro Español w Madrycie. Najstarszy teatr w Europie 1583-2026
10. Piąty element – MIŁOŚĆ. Wystawa Krakowskiego Klubu Fotograficznego

Czas wykonania (wątki): 0.69s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [3]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [4]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 14 procesach (rdzeniach)...
Zakończono w czasie 0.29s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [5]:
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"
NUMBER_OF_FACTS = 20

def fetch_single_fact():
    response = requests.get(CAT_API_URL)
    return response.json().get('fact')

def fetch_sequential(n):
    print("Rozpoczęto pobieranie sekwencyjne...")
    start_time = time.time()

    facts = []
    for _ in range(n):
        facts.append(fetch_single_fact())

    execution_time = time.time() - start_time
    print(f"Czas (sekwencyjnie): {execution_time:.4f} sekund")
    return facts

def fetch_concurrent(n):
    print("Rozpoczęto pobieranie wielowątkowe...")
    start_time = time.time()

    facts = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        futures = [executor.submit(fetch_single_fact) for _ in range(n)]

        for future in concurrent.futures.as_completed(futures):
            facts.append(future.result())

    execution_time = time.time() - start_time
    print(f"Czas (wielowątkowo): {execution_time:.4f} sekund")
    return facts

if __name__ == "__main__":
    print(f"Pobieranie {NUMBER_OF_FACTS} faktów o kotach...\n")

    seq_results = fetch_sequential(NUMBER_OF_FACTS)

    print("-" * 30)

    conc_results = fetch_concurrent(NUMBER_OF_FACTS)

    print("-" * 30)

    print(f"Przykładowy fakt: {conc_results[0]}")

Pobieranie 20 faktów o kotach...

Rozpoczęto pobieranie sekwencyjne...
Czas (sekwencyjnie): 5.4214 sekund
------------------------------
Rozpoczęto pobieranie wielowątkowe...
Czas (wielowątkowo): 0.7863 sekund
------------------------------
Przykładowy fakt: Cats can be taught to walk on a leash, but a lot of time and patience is required to teach them. The younger the cat is, the easier it will be for them to learn.


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [6]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

def producer(q, limit):
    print("[Producent] Rozpoczynam generowanie...")
    for i in range(1, limit + 1):
        q.put(i)
        print(f"[Producent] Dodano do kolejki: {i}")
        time.sleep(0.1)

    q.put(None)
    print("[Producent] Zakończył pracę.")

def consumer_even(q):
    while True:
        item = q.get()

        if item is None:
            q.put(None)
            break

        if item % 2 == 0:
            print(f"  --> [Konsument 1] Przetworzono PARZYSTĄ: {item}")
        else:
            q.put(item)
            time.sleep(0.05)

def consumer_odd(q):
    while True:
        item = q.get()

        if item is None:
            q.put(None)
            break

        if item % 2 != 0:
            print(f"  --> [Konsument 2] Przetworzono NIEPARZYSTĄ: {item}")
        else:
            q.put(item)
            time.sleep(0.05)

if __name__ == "__main__":
    LIMIT_LICZB = 10
    wspoldzielona_kolejka = queue.Queue()

    t_producer = threading.Thread(target=producer, args=(wspoldzielona_kolejka, LIMIT_LICZB))
    t_consumer_1 = threading.Thread(target=consumer_even, args=(wspoldzielona_kolejka,))
    t_consumer_2 = threading.Thread(target=consumer_odd, args=(wspoldzielona_kolejka,))

    t_consumer_1.start()
    t_consumer_2.start()
    t_producer.start()

    t_producer.join()
    t_consumer_1.join()
    t_consumer_2.join()

    print("Program zakończył działanie pomyślnie.")

[Producent] Rozpoczynam generowanie...
[Producent] Dodano do kolejki: 1
  --> [Konsument 2] Przetworzono NIEPARZYSTĄ: 1
[Producent] Dodano do kolejki: 2
  --> [Konsument 1] Przetworzono PARZYSTĄ: 2
[Producent] Dodano do kolejki: 3
  --> [Konsument 2] Przetworzono NIEPARZYSTĄ: 3
[Producent] Dodano do kolejki: 4
  --> [Konsument 1] Przetworzono PARZYSTĄ: 4
[Producent] Dodano do kolejki: 5  --> [Konsument 2] Przetworzono NIEPARZYSTĄ: 5

[Producent] Dodano do kolejki: 6
  --> [Konsument 1] Przetworzono PARZYSTĄ: 6
[Producent] Dodano do kolejki: 7  --> [Konsument 2] Przetworzono NIEPARZYSTĄ: 7

[Producent] Dodano do kolejki: 8
  --> [Konsument 1] Przetworzono PARZYSTĄ: 8
[Producent] Dodano do kolejki: 9  --> [Konsument 2] Przetworzono NIEPARZYSTĄ: 9

[Producent] Dodano do kolejki: 10
  --> [Konsument 1] Przetworzono PARZYSTĄ: 10
[Producent] Zakończył pracę.
Program zakończył działanie pomyślnie.


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [7]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    zakres_liczb = range(1, 10001)

    liczba_procesow = multiprocessing.cpu_count()

    print(f"Rozpoczynam obliczenia przy użyciu {liczba_procesow} procesów...")
    start_time = time.time()

    with multiprocessing.Pool(processes=liczba_procesow) as pool:
        wyniki = pool.map(calculate_power_sum, zakres_liczb)

    czas_wykonania = time.time() - start_time

    print("Obliczenia zakończone pomyślnie.")
    print(f"Przetworzono {len(wyniki)} elementów.")
    print(f"Czas wykonania: {czas_wykonania:.4f} sekund")

Rozpoczynam obliczenia przy użyciu 14 procesów...
Obliczenia zakończone pomyślnie.
Przetworzono 10000 elementów.
Czas wykonania: 0.1395 sekund
